---

In [1]:
!pwd

/home/amanr.mds2024/Developer/AML/GenAI_proj


In [2]:
%%bash
source ~/miniconda3/etc/profile.d/conda.sh
conda activate AML-GenAI_proj

In [3]:
%%bash
# Start Redis Stack Server in the background as a daemon
./redis-stack-server-7.2.0-v9/bin/redis-stack-server --daemonize yes

echo "Redis Stack is running on port 6379"

Starting redis-stack-server, database path ./redis-stack-server-7.2.0-v9/var/db/redis-stack


1903599:C 20 Apr 2026 15:33:24.769 # WARNING Memory overcommit must be enabled! Without it, a background save or replication may fail under low memory condition. Being disabled, it can also cause failures without low memory condition, see https://github.com/jemalloc/jemalloc/issues/1328. To fix this issue add 'vm.overcommit_memory = 1' to /etc/sysctl.conf and then reboot or run the command 'sysctl vm.overcommit_memory=1' for this to take effect.
Redis Stack is running on port 6379


In [4]:
%%bash
cd /home/amanr.mds2024/Developer/AML/GenAI_proj

# Start MLflow in the background and pipe output to a log file
nohup mlflow server --host 127.0.0.1 --port 5000 > mlflow.log 2>&1 &

echo "MLflow tracking server is running on http://127.0.0.1:5000"

MLflow tracking server is running on http://127.0.0.1:5000


In [5]:
%%bash
cd /home/amanr.mds2024/Developer/AML/GenAI_proj

export REDIS_URL="redis://localhost:6379"
export MLFLOW_TRACKING_URI="http://localhost:5000"

# Start the FastAPI app in the background
nohup uvicorn src.api.main:app --host 0.0.0.0 --port 8000 > api.log 2>&1 &

echo "FastAPI backend is running on http://localhost:8000"

FastAPI backend is running on http://localhost:8000


In [6]:
%%bash
cd /home/amanr.mds2024/Developer/AML/GenAI_proj

# Start Streamlit in the background
nohup streamlit run src/app/ui.py --server.port=8501 --server.address=0.0.0.0 > ui.log 2>&1 &

echo "Streamlit UI is running on http://localhost:8501"

Streamlit UI is running on http://localhost:8501


In [7]:
!cat api.log

---

In [4]:
%%bash
pwd
python -m src.model_training.train_ddp

/home/amanr.mds2024/Developer/AML/GenAI_proj


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7361.66it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Statu

--- Starting DDP Training on 2 GPUs ---
Epoch 1 complete. Loss: 1.3965
Epoch 2 complete. Loss: 1.3347
Epoch 3 complete. Loss: 1.2812


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.17it/s]


Model successfully saved to /home/amanr.mds2024/Developer/AML/GenAI_proj/models/distilled_query_model


---

In [5]:
%%bash
cd /home/amanr.mds2024/Developer/AML/GenAI_proj
source ~/miniconda3/etc/profile.d/conda.sh
conda activate AML-GenAI_proj

# 1. Clean up lingering background processes
pkill -f "redis-stack-server"
pkill -f "mlflow server"
pkill -f "uvicorn src.api.main:app"
pkill -f "streamlit run"

# 2. Start Redis Stack Server
./redis-stack-server-7.2.0-v9/bin/redis-stack-server --daemonize yes
echo "Redis Stack is running on port 6379"

# 3. Start MLflow Tracking Server
nohup mlflow server --host 127.0.0.1 --port 5000 > mlflow.log 2>&1 &
echo "MLflow tracking server is running on http://127.0.0.1:5000"

# 4. Give MLflow 3 seconds to fully initialize before starting FastAPI
sleep 3

# 5. Export variables strictly as IPv4 to avoid resolution errors
export REDIS_URL="redis://127.0.0.1:6379"
export MLFLOW_TRACKING_URI="http://127.0.0.1:5000"

# 6. Start FastAPI Backend
nohup uvicorn src.api.main:app --host 0.0.0.0 --port 8000 > api.log 2>&1 &
echo "FastAPI backend is running on http://0.0.0.0:8000"

# 7. Start Streamlit UI
nohup streamlit run src/app/ui.py --server.port=8501 --server.address=0.0.0.0 > ui.log 2>&1 &
echo "Streamlit UI is running on http://0.0.0.0:8501"

Starting redis-stack-server, database path ./redis-stack-server-7.2.0-v9/var/db/redis-stack
1854808:C 20 Apr 2026 11:41:40.973 # WARNING Memory overcommit must be enabled! Without it, a background save or replication may fail under low memory condition. Being disabled, it can also cause failures without low memory condition, see https://github.com/jemalloc/jemalloc/issues/1328. To fix this issue add 'vm.overcommit_memory = 1' to /etc/sysctl.conf and then reboot or run the command 'sysctl vm.overcommit_memory=1' for this to take effect.
Redis Stack is running on port 6379
MLflow tracking server is running on http://127.0.0.1:5000
FastAPI backend is running on http://0.0.0.0:8000
Streamlit UI is running on http://0.0.0.0:8501
